<a href="https://colab.research.google.com/github/a7madtalal123/Lightweight-Graph-Embedding-Augmentation-for-Airport-Traffic-Forecasting/blob/main/airport_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- CELL 1 · Setup -----------------------------------------------------------
# HOW TO RUN IN COLAB:
#   First run  → installs packages → runtime restarts automatically
#   Second run → skips install     → completes setup (imports + config)
#   Simply run this cell twice if you see a restart message.

from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os

# -- Install packages (auto-skipped if already present) -----------------------
try:
    import node2vec, shap, xgboost, networkx, lightgbm
    print("Packages already installed - skipping.")
except ImportError:
    print("Installing packages. Runtime will restart automatically ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.26.4"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "node2vec", "shap", "statsmodels", "xgboost", "lightgbm",
        "scikit-learn", "pandas", "networkx", "pyarrow"])
    print("Install complete. Restarting runtime now ...")
    os.kill(os.getpid(), 9)

# -- Imports ------------------------------------------------------------------
import warnings, random, pickle, urllib.request
from math import sqrt
from itertools import product

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
from node2vec import Node2Vec
import shap
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
torch.set_num_threads(4)

# -- Configuration ------------------------------------------------------------
DRIVE_BASE  = "/content/drive/MyDrive/papers/airport"
DATA_PATH   = f"{DRIVE_BASE}/flights_RUH.parquet"
OUTPUT_DIR  = f"{DRIVE_BASE}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEEDS       = [42, 123, 456, 789, 2024]
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

RUH_IATA    = "RUH"
RUH_ICAO    = "OERK"

N2V_DIM     = 64
N2V_WALK    = 80
N2V_WALKS   = 10
N2V_P       = 1.0
N2V_Q       = 0.5
N2V_WINDOW  = 10
N2V_EPOCHS  = 5
N2V_LR      = 0.025

SEQ_LEN     = 24
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- Helper functions ---------------------------------------------------------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def smape(y, yhat):
    y, yhat = np.array(y, float), np.array(yhat, float)
    denom = (np.abs(y) + np.abs(yhat)) / 2.0
    denom[denom == 0] = 1e-8
    return float(np.mean(np.abs(y - yhat) / denom) * 100)

def metrics(y, yhat):
    return {
        "MAE":   round(mean_absolute_error(y, yhat), 4),
        "RMSE":  round(sqrt(mean_squared_error(y, yhat)), 4),
        "sMAPE": round(smape(y, yhat), 4),
    }

print(f"\nSetup complete | device={DEVICE} | outputs -> {OUTPUT_DIR}")
print(f"numpy {np.__version__} | torch {torch.__version__}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Packages already installed - skipping.

Setup complete | device=cuda | outputs -> /content/drive/MyDrive/papers/airport/outputs
numpy 1.26.4 | torch 2.11.0+cu128


In [2]:
# ─── CELL 2 · Preprocess ───────────────────────────────────────────────────

# ── Load raw data ────────────────────────────────────────────────────────────
df = pd.read_parquet(DATA_PATH)
print(f"Loaded  {len(df):,} records  |  {df.shape[1]} columns")

# Normalise column names to lowercase for safety
df.columns = [c.strip() for c in df.columns]

# ── Parse scheduled datetime ─────────────────────────────────────────────────
for col in ["movement.scheduledTime.local", "movement.scheduledTime.utc"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "movement.scheduledTime.local" in df.columns:
    df["scheduled_dt"] = df["movement.scheduledTime.local"].fillna(
        df.get("movement.scheduledTime.utc"))
else:
    df["scheduled_dt"] = df["movement.scheduledTime.utc"]

df = df[df["scheduled_dt"].notna()].copy()

# ── Remove cancelled flights ─────────────────────────────────────────────────
if "status" in df.columns:
    mask_cancel = df["status"].astype(str).str.lower().str.contains("cancel", na=False)
    df = df[~mask_cancel].copy()
    print(f"After removing cancelled: {len(df):,} records")

# ── Identify departures & arrivals ───────────────────────────────────────────
if "flight_type" in df.columns:
    df["_is_dep"] = df["flight_type"].astype(str).str.lower().str.contains("dep")
    df["_is_arr"] = df["flight_type"].astype(str).str.lower().str.contains("arr")
else:
    df["_is_dep"] = df["origin_airport_iata"].astype(str).str.upper() == RUH_IATA
    df["_is_arr"] = df["destination_airport_iata"].astype(str).str.upper() == RUH_IATA

dep_df = df[df["_is_dep"]].copy()
arr_df = df[df["_is_arr"]].copy()
print(f"Departures: {len(dep_df):,}  |  Arrivals: {len(arr_df):,}")

# ── Hourly floor ─────────────────────────────────────────────────────────────
df["date_hour"] = df["scheduled_dt"].dt.floor("h")
dep_df["date_hour"] = dep_df["scheduled_dt"].dt.floor("h")
arr_df["date_hour"] = arr_df["scheduled_dt"].dt.floor("h")

# ── Airport-level total hourly movements ────────────────────────────────────
ts = df.groupby("date_hour").size().reset_index(name="y")
ts = ts.sort_values("date_hour").reset_index(drop=True)

# Fill complete hourly range (retain zero-movement hours)
full_idx = pd.date_range(ts["date_hour"].min(), ts["date_hour"].max(), freq="h")
ts = (ts.set_index("date_hour")
        .reindex(full_idx, fill_value=0)
        .reset_index()
        .rename(columns={"index": "date_hour"}))

print(f"Hourly series: {len(ts):,} hours  "
      f"|  {ts['date_hour'].min().date()} → {ts['date_hour'].max().date()}")
print(f"Mean: {ts['y'].mean():.2f}  |  Max: {ts['y'].max()}  |  Zeros: {(ts['y']==0).sum()}")

# ── Temporal features ────────────────────────────────────────────────────────
ts["hour"]       = ts["date_hour"].dt.hour
ts["weekday"]    = ts["date_hour"].dt.weekday
ts["month"]      = ts["date_hour"].dt.month
ts["is_weekend"] = ts["weekday"].isin([4, 5]).astype(int)   # Fri/Sat (Saudi calendar)
ts["hour_sin"]   = np.sin(2 * np.pi * ts["hour"] / 24)
ts["hour_cos"]   = np.cos(2 * np.pi * ts["hour"] / 24)

# Lag features
ts["lag1"] = ts["y"].shift(1)
ts["lag3"] = ts["y"].shift(3)
ts["lag6"] = ts["y"].shift(6)

# Rolling mean features (shifted to avoid leakage)
ts["ma3"] = ts["y"].rolling(3).mean().shift(1)
ts["ma6"] = ts["y"].rolling(6).mean().shift(1)

ts = ts.dropna().reset_index(drop=True)
print(f"After feature engineering: {len(ts):,} rows")

# ── 70 / 15 / 15 chronological split ────────────────────────────────────────
n        = len(ts)
tr_end   = int(n * TRAIN_RATIO)
val_end  = int(n * (TRAIN_RATIO + VAL_RATIO))

split_info = {
    "train_rows": tr_end,
    "val_rows":   val_end - tr_end,
    "test_rows":  n - val_end,
    "train_start": str(ts["date_hour"].iloc[0].date()),
    "train_end":   str(ts["date_hour"].iloc[tr_end - 1].date()),
    "val_start":   str(ts["date_hour"].iloc[tr_end].date()),
    "val_end":     str(ts["date_hour"].iloc[val_end - 1].date()),
    "test_start":  str(ts["date_hour"].iloc[val_end].date()),
    "test_end":    str(ts["date_hour"].iloc[-1].date()),
}
for k, v in split_info.items():
    print(f"  {k}: {v}")

# ── Save ─────────────────────────────────────────────────────────────────────
ts.to_csv(f"{OUTPUT_DIR}/hourly_features.csv", index=False)
dep_df.to_csv(f"{OUTPUT_DIR}/departures.csv", index=False)   # needed for graph features
pd.DataFrame([split_info]).to_csv(f"{OUTPUT_DIR}/split_info.csv", index=False)
print(f"\n✅  Saved hourly_features.csv  |  departures.csv  |  split_info.csv")


Loaded  153,308 records  |  23 columns
After removing cancelled: 153,197 records
Departures: 78,193  |  Arrivals: 75,004
Hourly series: 5,028 hours  |  2025-03-15 → 2025-10-10
Mean: 30.47  |  Max: 48  |  Zeros: 0
After feature engineering: 5,022 rows
  train_rows: 3515
  val_rows: 753
  test_rows: 754
  train_start: 2025-03-15
  train_end: 2025-08-08
  val_start: 2025-08-08
  val_end: 2025-09-09
  test_start: 2025-09-09
  test_end: 2025-10-10

✅  Saved hourly_features.csv  |  departures.csv  |  split_info.csv


In [3]:
# ─── CELL 3 · Build Global Airport Graph ───────────────────────────────────

G = nx.DiGraph()

# ── Step 1: RUH direct connections from dataset ──────────────────────────────
dep_df = pd.read_csv(f"{OUTPUT_DIR}/departures.csv")

def safe_iata(val):
    s = str(val).strip().upper()
    return s if (len(s) == 3 and s.isalpha() and s != "NAN") else None

# Departures: RUH → destination
for dest, grp in dep_df.groupby("destination_airport_iata"):
    d = safe_iata(dest)
    if d and d != RUH_IATA:
        w = len(grp)
        if G.has_edge(RUH_IATA, d):
            G[RUH_IATA][d]["weight"] += w
        else:
            G.add_edge(RUH_IATA, d, weight=w)

# Arrivals: read back from raw df (origin → RUH)
df_raw = pd.read_parquet(DATA_PATH)
if "flight_type" in df_raw.columns:
    arr_raw = df_raw[df_raw["flight_type"].astype(str).str.lower().str.contains("arr")].copy()
else:
    arr_raw = df_raw[df_raw["destination_airport_iata"].astype(str).str.upper() == RUH_IATA].copy()

for orig, grp in arr_raw.groupby("origin_airport_iata"):
    o = safe_iata(orig)
    if o and o != RUH_IATA:
        w = len(grp)
        if G.has_edge(o, RUH_IATA):
            G[o][RUH_IATA]["weight"] += w
        else:
            G.add_edge(o, RUH_IATA, weight=w)

ruh_nodes = G.number_of_nodes()
ruh_edges = G.number_of_edges()
ruh_neighbors = set(G.successors(RUH_IATA)) | set(G.predecessors(RUH_IATA))
print(f"RUH local graph  →  {ruh_nodes} nodes, {ruh_edges} edges")
print(f"RUH direct neighbours: {len(ruh_neighbors)}")

# ── Step 2: Download OpenFlights global routes ───────────────────────────────
print("\nDownloading OpenFlights global routes ...")
of_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"
try:
    resp = urllib.request.urlopen(of_url, timeout=30)
    routes_raw = resp.read().decode("utf-8", errors="replace")
    print(f"  Downloaded {len(routes_raw):,} bytes")
except Exception as e:
    print(f"  ⚠ Download failed ({e}). Using only RUH local graph.")
    routes_raw = ""

# Format: Airline,AirlineID,Src,SrcID,Dst,DstID,Codeshare,Stops,Equipment
added = 0
if routes_raw:
    for line in routes_raw.strip().split("\n"):
        parts = line.split(",")
        if len(parts) < 5:
            continue
        src, dst = parts[2].strip().upper(), parts[4].strip().upper()
        if (len(src) != 3 or len(dst) != 3 or
                not src.isalpha() or not dst.isalpha() or
                src == "\\N" or dst == "\\N"):
            continue
        # Keep only routes involving RUH neighbours (2-hop)
        if src in ruh_neighbors or dst in ruh_neighbors or src == RUH_IATA or dst == RUH_IATA:
            if not G.has_edge(src, dst):
                G.add_edge(src, dst, weight=1)
                added += 1

print(f"  Added {added:,} OpenFlights 2-hop edges")

total_nodes = G.number_of_nodes()
total_edges = G.number_of_edges()
print(f"\nFinal global graph  →  {total_nodes:,} nodes, {total_edges:,} edges")

# ── Graph stats ───────────────────────────────────────────────────────────────
degrees = dict(G.degree())
top5 = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
print(f"Top-5 by degree: {top5}")
print(f"RUH degree: {degrees.get(RUH_IATA, 0)}")
print(f"Mean degree: {np.mean(list(degrees.values())):.1f}")

stats_df = pd.DataFrame({
    "airport":       list(degrees.keys()),
    "degree":        list(degrees.values()),
    "in_degree":     [G.in_degree(n) for n in degrees],
    "out_degree":    [G.out_degree(n) for n in degrees],
    "is_ruh_direct": [n in ruh_neighbors or n == RUH_IATA for n in degrees],
})
stats_df.to_csv(f"{OUTPUT_DIR}/graph_stats.csv", index=False)

# Save graph object
with open(f"{OUTPUT_DIR}/airport_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print(f"\n✅  Saved graph_stats.csv  |  airport_graph.pkl")


RUH local graph  →  113 nodes, 112 edges
RUH direct neighbours: 112

  Downloaded 2,377,148 bytes
  Added 11,088 OpenFlights 2-hop edges

Final global graph  →  1,198 nodes, 11,200 edges
Top-5 by degree: [('FRA', 477), ('CDG', 470), ('IST', 457), ('PEK', 413), ('MUC', 380)]
RUH degree: 203
Mean degree: 18.7

✅  Saved graph_stats.csv  |  airport_graph.pkl


In [4]:
# ─── CELL 4 · Node2Vec + DeepWalk Embeddings + Hourly Graph Features ──────
import time as _time

# ── Load graph ────────────────────────────────────────────────────────────────
with open(f"{OUTPUT_DIR}/airport_graph.pkl", "rb") as f:
    G = pickle.load(f)

ruh_neighbors = set(G.successors(RUH_IATA)) | set(G.predecessors(RUH_IATA))

# ── Node2Vec (p=1.0, q=0.5 – biased walk) ────────────────────────────────────
print("Running Node2Vec (p=1.0, q=0.5) ...")
set_seed(42)
t0 = _time.time()
n2v = Node2Vec(
    G,
    dimensions  = N2V_DIM,
    walk_length = N2V_WALK,
    num_walks   = N2V_WALKS,
    p           = N2V_P,
    q           = N2V_Q,
    weight_key  = "weight",
    workers     = 4,
    quiet       = True,
)
model_n2v = n2v.fit(
    window      = N2V_WINDOW,
    min_count   = 1,
    batch_words = 64,
    epochs      = N2V_EPOCHS,
    alpha       = N2V_LR,
)
t_n2v = _time.time() - t0
print(f"Node2Vec done in {t_n2v:.1f}s")

N2V_COLS = [f"n2v_{i}" for i in range(N2V_DIM)]
emb_df = pd.DataFrame(
    [{**{"airport": node}, **{f"n2v_{i}": v for i, v in enumerate(model_n2v.wv[node])}}
     for node in G.nodes() if node in model_n2v.wv]
)
emb_df.to_csv(f"{OUTPUT_DIR}/node2vec_embeddings.csv", index=False)
print(f"Node2Vec embeddings: {len(emb_df):,} airports  |  dim={N2V_DIM}")

# ── DeepWalk (p=1.0, q=1.0 – unbiased walk, same hyperparameters) ────────────
print("\nRunning DeepWalk (p=1.0, q=1.0) ...")
set_seed(42)
t0 = _time.time()
dw = Node2Vec(
    G,
    dimensions  = N2V_DIM,
    walk_length = N2V_WALK,
    num_walks   = N2V_WALKS,
    p           = 1.0,
    q           = 1.0,
    weight_key  = "weight",
    workers     = 4,
    quiet       = True,
)
model_dw = dw.fit(
    window      = N2V_WINDOW,
    min_count   = 1,
    batch_words = 64,
    epochs      = N2V_EPOCHS,
    alpha       = N2V_LR,
)
t_dw = _time.time() - t0
print(f"DeepWalk done in {t_dw:.1f}s")

DW_COLS = [f"dw_{i}" for i in range(N2V_DIM)]
dw_df = pd.DataFrame(
    [{**{"airport": node}, **{f"dw_{i}": v for i, v in enumerate(model_dw.wv[node])}}
     for node in G.nodes() if node in model_dw.wv]
)
dw_df.to_csv(f"{OUTPUT_DIR}/deepwalk_embeddings.csv", index=False)
print(f"DeepWalk embeddings: {len(dw_df):,} airports  |  dim={N2V_DIM}")

# Save embedding timing
pd.DataFrame([
    {"method": "Node2Vec", "embedding_time_s": round(t_n2v, 2)},
    {"method": "DeepWalk", "embedding_time_s": round(t_dw,  2)},
]).to_csv(f"{OUTPUT_DIR}/timing_embedding.csv", index=False)
print(f"\nEmbedding times: Node2Vec={t_n2v:.1f}s  |  DeepWalk={t_dw:.1f}s")

# ── Centrality features (for ablation) ────────────────────────────────────────
print("\nComputing centrality features ...")
pagerank  = nx.pagerank(G, weight="weight", max_iter=200)
in_deg_c  = nx.in_degree_centrality(G)
out_deg_c = nx.out_degree_centrality(G)

sub_nodes = ruh_neighbors | {RUH_IATA}
sub_G     = G.subgraph(sub_nodes).copy()
between_c = nx.betweenness_centrality(sub_G, weight="weight", normalized=True)

CENT_COLS = ["pagerank", "in_deg_c", "out_deg_c", "betweenness"]
cent_df   = pd.DataFrame({
    "airport":     list(pagerank.keys()),
    "pagerank":    [pagerank[n]          for n in pagerank],
    "in_deg_c":    [in_deg_c[n]          for n in pagerank],
    "out_deg_c":   [out_deg_c[n]         for n in pagerank],
    "betweenness": [between_c.get(n, 0)  for n in pagerank],
})
cent_df.to_csv(f"{OUTPUT_DIR}/centrality_features.csv", index=False)

# ── Traffic-weighted mean graph features per hour ─────────────────────────────
print("\nBuilding hourly graph features ...")
ts  = pd.read_csv(f"{OUTPUT_DIR}/hourly_features.csv",  parse_dates=["date_hour"])
dep = pd.read_csv(f"{OUTPUT_DIR}/departures.csv",       parse_dates=["date_hour"])

if "date_hour" not in dep.columns:
    dep["scheduled_dt"] = pd.to_datetime(dep.get(
        "movement.scheduledTime.local",
        dep.get("movement.scheduledTime.utc")), errors="coerce")
    dep["date_hour"] = dep["scheduled_dt"].dt.floor("h")

dep["dest_iata"] = dep["destination_airport_iata"].astype(str).str.upper()
dep = dep[dep["dest_iata"].apply(lambda x: len(x) == 3 and x.isalpha())]

# Merge Node2Vec, DeepWalk, and centrality for each departure
dep = dep.merge(emb_df.rename(columns={"airport": "dest_iata"}),  on="dest_iata", how="left")
dep = dep.merge(dw_df.rename(columns={"airport":  "dest_iata"}),  on="dest_iata", how="left")
dep = dep.merge(cent_df.rename(columns={"airport": "dest_iata"}), on="dest_iata", how="left")

GRAPH_FEAT_COLS = N2V_COLS + DW_COLS + CENT_COLS

def hourly_mean(group):
    result = {}
    for col in GRAPH_FEAT_COLS:
        vals = group[col].fillna(0).values
        result[col] = float(np.mean(vals)) if len(vals) > 0 else 0.0
    return pd.Series(result)

hourly_graph = dep.groupby("date_hour").apply(hourly_mean).reset_index()
ts = ts.merge(hourly_graph, on="date_hour", how="left")
for col in GRAPH_FEAT_COLS:
    ts[col] = ts[col].fillna(0.0)

ts.to_csv(f"{OUTPUT_DIR}/hourly_graph_features.csv", index=False)
print(f"Feature columns: {len(GRAPH_FEAT_COLS)} graph + 10 temporal = {len(GRAPH_FEAT_COLS)+10} total")
print(f"\n✅  Saved node2vec_embeddings.csv | deepwalk_embeddings.csv | centrality_features.csv")
print(f"✅  Saved hourly_graph_features.csv | timing_embedding.csv")

Running Node2Vec (p=1.0, q=0.5) ...
Node2Vec done in 18.3s
Node2Vec embeddings: 1,198 airports  |  dim=64

Running DeepWalk (p=1.0, q=1.0) ...
DeepWalk done in 17.1s
DeepWalk embeddings: 1,198 airports  |  dim=64

Embedding times: Node2Vec=18.3s  |  DeepWalk=17.1s

Computing centrality features ...

Building hourly graph features ...
Feature columns: 132 graph + 10 temporal = 142 total

✅  Saved node2vec_embeddings.csv | deepwalk_embeddings.csv | centrality_features.csv
✅  Saved hourly_graph_features.csv | timing_embedding.csv


In [7]:
# ─── CELL 5 · Train All Models ─────────────────────────────────────────────

# ── Load data ─────────────────────────────────────────────────────────────────
ts = pd.read_csv(f"{OUTPUT_DIR}/hourly_graph_features.csv", parse_dates=["date_hour"])
ts = ts.sort_values("date_hour").reset_index(drop=True)

N2V_COLS  = [f"n2v_{i}" for i in range(N2V_DIM)]
DW_COLS   = [f"dw_{i}"  for i in range(N2V_DIM)]
CENT_COLS = ["pagerank", "in_deg_c", "out_deg_c", "betweenness"]
TEMP_COLS = ["lag1","lag3","lag6","ma3","ma6",
             "hour_sin","hour_cos","weekday","is_weekend","month"]
TARGET    = "y"

# Embedding sets: each entry is (suffix, feature columns)
EMBEDDING_SETS = {
    "N2V": TEMP_COLS + N2V_COLS,   # Node2Vec augmented (proposed)
    "DW":  TEMP_COLS + DW_COLS,    # DeepWalk augmented (comparison)
}

n       = len(ts)
tr_end  = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df    = ts.iloc[:tr_end].copy()
val_df      = ts.iloc[tr_end:val_end].copy()
test_df     = ts.iloc[val_end:].copy()
trainval_df = ts.iloc[:val_end].copy()

y_test     = test_df[TARGET].values
dates_test = test_df["date_hour"].values

# ── PyTorch models ────────────────────────────────────────────────────────────
class MLPNet(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

class LSTMNet(nn.Module):
    def __init__(self, in_dim, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def make_sequences(data_arr, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data_arr)):
        X.append(data_arr[i - seq_len:i, :-1])
        y.append(data_arr[i, -1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def train_torch(model, X_tr, y_tr, X_val, y_val,
                lr=1e-3, batch=64, max_epochs=150, patience=15):
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val = float("inf"); best_w = None; wait = 0
    tr_ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    loader = DataLoader(tr_ds, batch_size=batch, shuffle=True)
    for ep in range(max_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_val).to(DEVICE)
            vl = loss_fn(model(xv), torch.tensor(y_val).to(DEVICE)).item()
        if vl < best_val:
            best_val = vl; best_w = {k: v.clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_w)
    return model

# ── Scalers ───────────────────────────────────────────────────────────────────
scaler_temp = StandardScaler()
scaler_temp.fit(trainval_df[TEMP_COLS])

LSTM_COLS   = ["hour_sin","hour_cos","weekday","is_weekend","month", TARGET]
scaler_lstm = StandardScaler()
scaler_lstm.fit(trainval_df[LSTM_COLS])

# ── Persistence baseline (deterministic) ──────────────────────────────────────
pred_persistence = test_df["lag1"].values
m = metrics(y_test, pred_persistence)
print(f"Persistence  →  MAE {m['MAE']:.4f}  RMSE {m['RMSE']:.4f}  sMAPE {m['sMAPE']:.2f}%")

# ── ARIMA (deterministic, AIC order selection) ────────────────────────────────
print("\nFitting ARIMA (AIC order selection) ...")
y_train_arima = train_df[TARGET].values
best_aic, best_order = np.inf, (1, 1, 1)
for p, d, q in product(range(4), range(2), range(3)):
    try:
        aic = ARIMA(y_train_arima, order=(p, d, q)).fit().aic
        if aic < best_aic:
            best_aic, best_order = aic, (p, d, q)
    except Exception:
        continue
print(f"  Best ARIMA order: {best_order}  AIC={best_aic:.2f}")
arima_model = ARIMA(np.concatenate([train_df[TARGET].values, val_df[TARGET].values]),
                    order=best_order).fit()
pred_arima = np.clip(arima_model.forecast(steps=len(test_df)), 0, None)
m = metrics(y_test, pred_arima)
print(f"ARIMA{best_order}  →  MAE {m['MAE']:.4f}  RMSE {m['RMSE']:.4f}  sMAPE {m['sMAPE']:.2f}%")

# ── Multi-seed training loop ──────────────────────────────────────────────────
# Baselines:         temporal features only (RF, XGBoost, LightGBM, SVR, Ridge, MLP, LSTM)
# Graph-augmented:   temporal + Node2Vec  (RF+N2V, XGBoost+N2V, LightGBM+N2V)
#                    temporal + DeepWalk  (RF+DW,  XGBoost+DW,  LightGBM+DW)

MODEL_NAMES = ["XGBoost","RF","LightGBM","SVR","Ridge","MLP","LSTM",
               "XGBoost+N2V","RF+N2V","LightGBM+N2V",
               "XGBoost+DW", "RF+DW",  "LightGBM+DW"]
results   = {m: [] for m in MODEL_NAMES}
preds_all = {m: [] for m in MODEL_NAMES}

for seed in SEEDS:
    print(f"\n── Seed {seed} ─────────────────────────────────")
    set_seed(seed)

    # ── Raw arrays (temporal only – baselines) ────────────────────────────────
    Xt_tr  = train_df[TEMP_COLS].values
    Xt_val = val_df[TEMP_COLS].values
    Xt_tv  = trainval_df[TEMP_COLS].values
    Xt_te  = test_df[TEMP_COLS].values
    y_tr   = train_df[TARGET].values
    y_val  = val_df[TARGET].values
    y_tv   = trainval_df[TARGET].values

    # Scaled temporal (for SVR, MLP, Ridge)
    Xs_tr  = scaler_temp.transform(Xt_tr)
    Xs_val = scaler_temp.transform(Xt_val)
    Xs_tv  = scaler_temp.transform(Xt_tv)
    Xs_te  = scaler_temp.transform(Xt_te)

    # ── XGBoost baseline ──────────────────────────────────────────────────────
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                       subsample=0.8, colsample_bytree=0.8,
                       reg_lambda=1, reg_alpha=1,
                       early_stopping_rounds=20,
                       eval_metric="rmse", random_state=seed, verbosity=0)
    xgb.fit(Xt_tr, y_tr, eval_set=[(Xt_val, y_val)], verbose=False)
    p = np.clip(xgb.predict(Xt_te), 0, None)
    preds_all["XGBoost"].append(p)
    m = metrics(y_test, p); results["XGBoost"].append(m)
    print(f"  XGBoost      MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── RF baseline ───────────────────────────────────────────────────────────
    rf = RandomForestRegressor(n_estimators=300, max_depth=10,
                               min_samples_leaf=2, n_jobs=-1, random_state=seed)
    rf.fit(Xt_tv, y_tv)
    p = np.clip(rf.predict(Xt_te), 0, None)
    preds_all["RF"].append(p)
    m = metrics(y_test, p); results["RF"].append(m)
    print(f"  RF           MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── LightGBM baseline ─────────────────────────────────────────────────────
    lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                              subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1, reg_alpha=1,
                              random_state=seed, verbose=-1)
    lgbm.fit(Xt_tr, y_tr, eval_set=[(Xt_val, y_val)],
             callbacks=[lgb.early_stopping(20, verbose=False),
                        lgb.log_evaluation(-1)])
    p = np.clip(lgbm.predict(Xt_te), 0, None)
    preds_all["LightGBM"].append(p)
    m = metrics(y_test, p); results["LightGBM"].append(m)
    print(f"  LightGBM     MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── SVR baseline ──────────────────────────────────────────────────────────
    svr = SVR(kernel="rbf", C=10, epsilon=0.1)
    svr.fit(Xs_tv, y_tv)
    p = np.clip(svr.predict(Xs_te), 0, None)
    preds_all["SVR"].append(p)
    m = metrics(y_test, p); results["SVR"].append(m)
    print(f"  SVR          MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── Ridge baseline ────────────────────────────────────────────────────────
    ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0])
    ridge.fit(Xs_tv, y_tv)
    p = np.clip(ridge.predict(Xs_te), 0, None)
    preds_all["Ridge"].append(p)
    m = metrics(y_test, p); results["Ridge"].append(m)
    print(f"  Ridge        MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── MLP baseline ──────────────────────────────────────────────────────────
    mlp_net = MLPNet(len(TEMP_COLS)).to(DEVICE)
    mlp_net = train_torch(mlp_net,
                          Xs_tr.astype(np.float32), y_tr.astype(np.float32),
                          Xs_val.astype(np.float32), y_val.astype(np.float32))
    mlp_net.eval()
    with torch.no_grad():
        p = mlp_net(torch.tensor(Xs_te.astype(np.float32)).to(DEVICE)).cpu().numpy()
    p = np.clip(p, 0, None)
    preds_all["MLP"].append(p)
    m = metrics(y_test, p); results["MLP"].append(m)
    print(f"  MLP          MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── LSTM baseline ─────────────────────────────────────────────────────────
    lstm_data_tv = scaler_lstm.transform(trainval_df[LSTM_COLS])
    lstm_data_te = scaler_lstm.transform(test_df[LSTM_COLS])
    tr_seq_X, tr_seq_y = make_sequences(lstm_data_tv[:tr_end], SEQ_LEN)
    va_seq_X, va_seq_y = make_sequences(lstm_data_tv[tr_end - SEQ_LEN:], SEQ_LEN)
    te_full    = np.vstack([lstm_data_tv[-SEQ_LEN:], lstm_data_te])
    te_seq_X, _ = make_sequences(te_full, SEQ_LEN)

    lstm_net = LSTMNet(tr_seq_X.shape[2]).to(DEVICE)
    lstm_net = train_torch(lstm_net, tr_seq_X, tr_seq_y, va_seq_X, va_seq_y)
    lstm_net.eval()
    with torch.no_grad():
        p_scaled = lstm_net(torch.tensor(te_seq_X).to(DEVICE)).cpu().numpy()
    dummy       = np.zeros((len(p_scaled), len(LSTM_COLS)))
    dummy[:,-1] = p_scaled
    p = np.clip(scaler_lstm.inverse_transform(dummy)[:,-1], 0, None)
    preds_all["LSTM"].append(p)
    m = metrics(y_test, p); results["LSTM"].append(m)
    print(f"  LSTM         MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

    # ── Graph-augmented models (Node2Vec and DeepWalk) ────────────────────────
    for emb_name, feat_cols in EMBEDDING_SETS.items():
        Xg_tr  = train_df[feat_cols].values
        Xg_val = val_df[feat_cols].values
        Xg_tv  = trainval_df[feat_cols].values
        Xg_te  = test_df[feat_cols].values

        # XGBoost + embedding
        xgb_emb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                subsample=0.8, colsample_bytree=0.8,
                                reg_lambda=1, reg_alpha=1,
                                early_stopping_rounds=20,
                                eval_metric="rmse", random_state=seed, verbosity=0)
        xgb_emb.fit(Xg_tr, y_tr, eval_set=[(Xg_val, y_val)], verbose=False)
        p = np.clip(xgb_emb.predict(Xg_te), 0, None)
        key = f"XGBoost+{emb_name}"
        preds_all[key].append(p)
        m = metrics(y_test, p); results[key].append(m)
        print(f"  {key:15s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

        # RF + embedding
        rf_emb = RandomForestRegressor(n_estimators=300, max_depth=10,
                                       min_samples_leaf=2, n_jobs=-1, random_state=seed)
        rf_emb.fit(Xg_tv, y_tv)
        p = np.clip(rf_emb.predict(Xg_te), 0, None)
        key = f"RF+{emb_name}"
        preds_all[key].append(p)
        m = metrics(y_test, p); results[key].append(m)
        print(f"  {key:15s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")

        # LightGBM + embedding
        lgbm_emb = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                      subsample=0.8, colsample_bytree=0.8,
                                      reg_lambda=1, reg_alpha=1,
                                      random_state=seed, verbose=-1)
        lgbm_emb.fit(Xg_tr, y_tr, eval_set=[(Xg_val, y_val)],
                     callbacks=[lgb.early_stopping(20, verbose=False),
                                lgb.log_evaluation(-1)])
        p = np.clip(lgbm_emb.predict(Xg_te), 0, None)
        key = f"LightGBM+{emb_name}"
        preds_all[key].append(p)
        m = metrics(y_test, p); results[key].append(m)
        print(f"  {key:15s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  sMAPE={m['sMAPE']:.2f}%")


# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("SUMMARY  (mean ± std across 5 seeds)")
print("="*65)
summary_rows = []
for model_name, res_list in results.items():
    for metric_name in ["MAE","RMSE","sMAPE"]:
        vals = [r[metric_name] for r in res_list]
        summary_rows.append({
            "Model":  model_name,
            "Metric": metric_name,
            "Mean":   round(np.mean(vals), 4),
            "Std":    round(np.std(vals),  4),
        })
summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot_table(index="Model", columns="Metric",
                               values=["Mean","Std"], aggfunc="first")
print(pivot.to_string())

# Add deterministic models
det_rows = []
for mname, preds in [("Persistence", pred_persistence), ("ARIMA", pred_arima)]:
    m = metrics(y_test, preds)
    for k in ["MAE","RMSE","sMAPE"]:
        det_rows.append({"Model": mname, "Metric": k, "Mean": m[k], "Std": 0.0})
summary_df = pd.concat([summary_df, pd.DataFrame(det_rows)], ignore_index=True)
summary_df.to_csv(f"{OUTPUT_DIR}/metrics_summary.csv", index=False)

# Mean predictions
mean_preds = {name: np.mean(np.vstack(plist), axis=0) for name, plist in preds_all.items()}
mean_preds["Persistence"] = pred_persistence
mean_preds["ARIMA"]       = pred_arima

pred_df = pd.DataFrame({"date_hour": dates_test, "actual": y_test})
for name, p in mean_preds.items():
    pred_df[name.replace("+","_")] = p
pred_df.to_csv(f"{OUTPUT_DIR}/predictions_all_models.csv", index=False)

print(f"\n✅  Saved metrics_summary.csv  |  predictions_all_models.csv")

# Feature columns map (used by Cell 6 for SHAP and timing)
FEAT_COLS_MAP = {
    **{f"{b}":      TEMP_COLS                  for b in ["XGBoost","RF","LightGBM","SVR","Ridge","MLP","LSTM","Persistence","ARIMA"]},
    **{f"{b}+N2V":  TEMP_COLS + N2V_COLS       for b in ["XGBoost","RF","LightGBM"]},
    **{f"{b}+DW":   TEMP_COLS + DW_COLS        for b in ["XGBoost","RF","LightGBM"]},
}

# Store for Cell 6
joblib.dump({"results":      results,
             "mean_preds":   mean_preds,
             "y_test":       y_test,
             "dates_test":   dates_test,
             "train_df":     train_df,
             "val_df":       val_df,
             "test_df":      test_df,
             "trainval_df":  trainval_df,
             "ts":           ts,
             "TEMP_COLS":    TEMP_COLS,
             "N2V_COLS":     N2V_COLS,
             "DW_COLS":      DW_COLS,
             "CENT_COLS":    CENT_COLS,
             "EMBEDDING_SETS": EMBEDDING_SETS,
             "FEAT_COLS_MAP":  FEAT_COLS_MAP},
            f"{OUTPUT_DIR}/cell5_state.pkl")
print("✅  Saved cell5_state.pkl for Cell 6")

Persistence  →  MAE 4.4138  RMSE 5.7662  sMAPE 15.61%

Fitting ARIMA (AIC order selection) ...
  Best ARIMA order: (3, 0, 1)  AIC=20894.08
ARIMA(3, 0, 1)  →  MAE 5.7775  RMSE 6.9440  sMAPE 19.48%

── Seed 42 ─────────────────────────────────
  XGBoost      MAE=2.3812  RMSE=3.2660  sMAPE=8.12%
  RF           MAE=2.0290  RMSE=2.9116  sMAPE=6.93%
  LightGBM     MAE=2.4520  RMSE=3.3284  sMAPE=8.42%
  SVR          MAE=2.4107  RMSE=3.2917  sMAPE=8.28%
  Ridge        MAE=3.4803  RMSE=4.7076  sMAPE=12.09%
  MLP          MAE=3.2322  RMSE=4.1164  sMAPE=10.98%
  LSTM         MAE=2.9339  RMSE=3.9131  sMAPE=10.32%
  XGBoost+N2V      MAE=1.9927  RMSE=2.8649  sMAPE=6.77%
  RF+N2V           MAE=1.8097  RMSE=2.4748  sMAPE=6.18%
  LightGBM+N2V     MAE=2.0490  RMSE=3.0652  sMAPE=6.98%
  XGBoost+DW       MAE=1.9924  RMSE=2.8685  sMAPE=6.80%
  RF+DW            MAE=1.7892  RMSE=2.4325  sMAPE=6.12%
  LightGBM+DW      MAE=1.9497  RMSE=2.8049  sMAPE=6.65%

── Seed 123 ─────────────────────────────────
  XGBoos

In [10]:
# ── DM test: RF+DW vs RF+N2V (direct embedding comparison) ──────────────────
print("\n── Direct Embedding Comparison (DM test) ──────────────────")
embedding_pairs = [
    ("RF+DW",        "RF+N2V"),
    ("XGBoost+DW",   "XGBoost+N2V"),
    ("LightGBM+DW",  "LightGBM+N2V"),
]
emb_dm_rows = []
for m1, m2 in embedding_pairs:
    stat, pval = dm_test(y_test, mean_preds[m1], mean_preds[m2])
    sig = pval < 0.05
    emb_dm_rows.append({"Model_1": m1, "Model_2": m2,
                         "DM_stat": round(stat, 4), "p_value": round(pval, 4),
                         "Significant(p<0.05)": sig})
    print(f"  DM  {m1} vs {m2:15s}  stat={stat:+.3f}  p={pval:.4f}  {'✓' if sig else '✗'}")

pd.DataFrame(emb_dm_rows).to_csv(f"{OUTPUT_DIR}/dm_embedding_comparison.csv", index=False)
print(f"✅  Saved dm_embedding_comparison.csv")


── Direct Embedding Comparison (DM test) ──────────────────
  DM  RF+DW vs RF+N2V           stat=-1.179  p=0.2385  ✗
  DM  XGBoost+DW vs XGBoost+N2V      stat=+1.079  p=0.2807  ✗
  DM  LightGBM+DW vs LightGBM+N2V     stat=-3.749  p=0.0002  ✓
✅  Saved dm_embedding_comparison.csv


In [11]:
# ─── CELL 6 · Evaluate: DM Test · Ablation · SHAP · Temporal Robustness ───
import joblib, scipy.stats as st, time as _time

state          = joblib.load(f"{OUTPUT_DIR}/cell5_state.pkl")
results        = state["results"]
mean_preds     = state["mean_preds"]
y_test         = state["y_test"]
dates_test     = state["dates_test"]
TEMP_COLS      = state["TEMP_COLS"]
N2V_COLS       = state["N2V_COLS"]
DW_COLS        = state["DW_COLS"]
CENT_COLS      = state["CENT_COLS"]
FEAT_COLS_MAP  = state["FEAT_COLS_MAP"]
EMBEDDING_SETS = state["EMBEDDING_SETS"]
N2V_ALL_COLS   = EMBEDDING_SETS["N2V"]   # TEMP_COLS + N2V_COLS
DW_ALL_COLS    = EMBEDDING_SETS["DW"]    # TEMP_COLS + DW_COLS
TARGET         = "y"

# Reload ts with all graph features (including DeepWalk added in Cell 4)
ts = pd.read_csv(f"{OUTPUT_DIR}/hourly_graph_features.csv", parse_dates=["date_hour"])
ts = ts.sort_values("date_hour").reset_index(drop=True)
n       = len(ts)
tr_end  = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))
train_df    = ts.iloc[:tr_end].copy()
val_df      = ts.iloc[tr_end:val_end].copy()
test_df     = ts.iloc[val_end:].copy()
trainval_df = ts.iloc[:val_end].copy()

# ── Identify best model ───────────────────────────────────────────────────────
def mean_mae(name):
    if name in results:
        return np.mean([r["MAE"] for r in results[name]])
    return mean_absolute_error(y_test, mean_preds[name])

best_model = min(mean_preds.keys(), key=mean_mae)
print(f"Best model: {best_model}  (MAE={mean_mae(best_model):.4f})")

# ── 1. Diebold-Mariano Test ───────────────────────────────────────────────────
def dm_test(actual, p1, p2):
    e1 = (actual - p1) ** 2
    e2 = (actual - p2) ** 2
    d  = e1 - e2
    d_bar  = d.mean()
    gamma0 = np.var(d, ddof=1)
    dm_stat = d_bar / np.sqrt(gamma0 / len(d) + 1e-12)
    p_val   = 2 * (1 - st.norm.cdf(abs(dm_stat)))
    return float(dm_stat), float(p_val)

proposed  = ["XGBoost+N2V", "RF+N2V", "LightGBM+N2V",
             "XGBoost+DW",  "RF+DW",  "LightGBM+DW"]
baselines = ["XGBoost", "RF", "LightGBM", "SVR", "Ridge",
             "MLP", "LSTM", "Persistence", "ARIMA"]

dm_rows = []
for prop in proposed:
    for base in baselines:
        stat, pval = dm_test(y_test, mean_preds[prop], mean_preds[base])
        dm_rows.append({"Proposed": prop, "Baseline": base,
                        "DM_stat": round(stat, 4), "p_value": round(pval, 4),
                        "Significant(p<0.05)": pval < 0.05})
        print(f"  DM  {prop} vs {base:12s}  stat={stat:+.3f}  p={pval:.4f}  {'✓' if pval<0.05 else '✗'}")

pd.DataFrame(dm_rows).to_csv(f"{OUTPUT_DIR}/dm_test_results.csv", index=False)
print(f"✅  Saved dm_test_results.csv")

# ── 2. Ablation Study (all 5 seeds, same hyperparameters as main experiment) ──
print("\nRunning ablation study (5 seeds × 11 feature sets × 2 models) ...")

ABLATION_SETS = {
    "Lags only":                 ["lag1", "lag3", "lag6"],
    "RollMeans only":            ["ma3", "ma6"],
    "Calendar only":             ["hour_sin", "hour_cos", "weekday", "is_weekend", "month"],
    "Centrality only":           CENT_COLS,
    "DeepWalk only":             DW_COLS,
    "Node2Vec only":             N2V_COLS,
    "Lags + RollMeans":          ["lag1", "lag3", "lag6", "ma3", "ma6"],
    "AllTemporal":               TEMP_COLS,
    "Centrality + AllTemporal":  TEMP_COLS + CENT_COLS,
    "DeepWalk + AllTemporal":    TEMP_COLS + DW_COLS,
    "Node2Vec + AllTemporal":    TEMP_COLS + N2V_COLS,
}

abl_rows = []
for seed in SEEDS:
    set_seed(seed)
    for feat_name, feat_cols in ABLATION_SETS.items():
        for mname in ["XGBoost", "RF"]:
            X_tr      = train_df[feat_cols].values
            X_val_abl = val_df[feat_cols].values
            y_tr      = train_df[TARGET].values
            y_val_abl = val_df[TARGET].values
            X_tv      = trainval_df[feat_cols].values
            y_tv      = trainval_df[TARGET].values
            X_te      = test_df[feat_cols].values

            if mname == "XGBoost":
                mdl = XGBRegressor(
                    n_estimators=500, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8,
                    reg_lambda=1, reg_alpha=1,
                    early_stopping_rounds=20,
                    eval_metric="rmse", random_state=seed, verbosity=0
                )
                mdl.fit(X_tr, y_tr, eval_set=[(X_val_abl, y_val_abl)], verbose=False)
            else:
                mdl = RandomForestRegressor(
                    n_estimators=300, max_depth=10,
                    min_samples_leaf=2, n_jobs=-1, random_state=seed
                )
                mdl.fit(X_tv, y_tv)

            p = np.clip(mdl.predict(X_te), 0, None)
            m = metrics(y_test, p)
            abl_rows.append({
                "FeatureSet": feat_name, "Model": mname, "Seed": seed,
                **m, "n_features": len(feat_cols)
            })
    print(f"  Seed {seed} done")

abl_df_raw = pd.DataFrame(abl_rows)
abl_df_raw.to_csv(f"{OUTPUT_DIR}/ablation_results_raw.csv", index=False)

# Compute mean ± std across seeds, preserving order
abl_summary = []
for feat_name in ABLATION_SETS:
    for mname in ["XGBoost", "RF"]:
        grp = abl_df_raw[(abl_df_raw["FeatureSet"] == feat_name) &
                         (abl_df_raw["Model"] == mname)]
        abl_summary.append({
            "FeatureSet":  feat_name,
            "Model":       mname,
            "MAE_mean":    round(grp["MAE"].mean(),  4),
            "MAE_std":     round(grp["MAE"].std(),   4),
            "RMSE_mean":   round(grp["RMSE"].mean(), 4),
            "RMSE_std":    round(grp["RMSE"].std(),  4),
            "n_features":  int(grp["n_features"].iloc[0]),
        })

abl_df = pd.DataFrame(abl_summary)
abl_df.to_csv(f"{OUTPUT_DIR}/ablation_results.csv", index=False)
print(f"\n✅  Saved ablation_results_raw.csv | ablation_results.csv")
print(abl_df.to_string(index=False))

# ── 3. SHAP for best model ────────────────────────────────────────────────────
print(f"\nSHAP analysis for best model: {best_model} ...")
set_seed(SEEDS[0])

best_feat_cols = FEAT_COLS_MAP[best_model]
X_tr_best = trainval_df[best_feat_cols].values
y_tr_best = trainval_df[TARGET].values
X_te_best = test_df[best_feat_cols].values

if "XGBoost" in best_model:
    best_mdl = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8,
                            reg_lambda=1, reg_alpha=1, random_state=SEEDS[0], verbosity=0)
    n_tv = int(len(X_tr_best) * 0.85)
    best_mdl.fit(X_tr_best[:n_tv], y_tr_best[:n_tv],
                 eval_set=[(X_tr_best[n_tv:], y_tr_best[n_tv:])],
                 early_stopping_rounds=20, verbose=False)
else:
    best_mdl = RandomForestRegressor(n_estimators=300, max_depth=10,
                                     min_samples_leaf=2, n_jobs=-1, random_state=SEEDS[0])
    best_mdl.fit(X_tr_best, y_tr_best)

explainer   = shap.TreeExplainer(best_mdl)
shap_values = explainer.shap_values(X_te_best)
pd.DataFrame(shap_values, columns=best_feat_cols).to_csv(
    f"{OUTPUT_DIR}/shap_values.csv", index=False)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top20 = pd.Series(mean_abs_shap, index=best_feat_cols).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot(kind="barh", ax=ax, color="#2196F3")
ax.set_xlabel("Mean |SHAP value|", fontsize=12)
ax.set_title(f"SHAP Feature Importance – {best_model}", fontsize=13)
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig_shap_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅  Saved shap_values.csv | fig_shap_importance.png")

# ── 4. Temporal robustness – rolling monthly MAE / RMSE ──────────────────────
print("\nTemporal robustness analysis ...")
ts["date_hour"]  = pd.to_datetime(ts["date_hour"])
ts["year_month"] = ts["date_hour"].dt.to_period("M")
months = sorted(ts["year_month"].unique())

robust_rows = []
for i, month in enumerate(months[3:], start=3):
    # Use the union of DW and N2V cols so both models can be trained per month
    shared_cols = DW_ALL_COLS + [c for c in N2V_ALL_COLS if c not in DW_ALL_COLS]
    tr = ts[ts["year_month"] < month].dropna(subset=shared_cols + [TARGET])
    te = ts[ts["year_month"] == month].dropna(subset=shared_cols + [TARGET])
    if len(tr) < 100 or len(te) < 10:
        continue

    for mname, feat_cols_r, params in [
        ("RF+DW",  DW_ALL_COLS,  {"n_estimators":200,"max_depth":10,"min_samples_leaf":2,"n_jobs":-1,"random_state":42}),
        ("RF+N2V", N2V_ALL_COLS, {"n_estimators":200,"max_depth":10,"min_samples_leaf":2,"n_jobs":-1,"random_state":42}),
    ]:
        X_tr_r = tr[feat_cols_r].values;  y_tr_r = tr[TARGET].values
        X_te_r = te[feat_cols_r].values;  y_te_r = te[TARGET].values
        mdl = RandomForestRegressor(**params)
        mdl.fit(X_tr_r, y_tr_r)
        p = np.clip(mdl.predict(X_te_r), 0, None)
        m = metrics(y_te_r, p)
        robust_rows.append({"Month": str(month), "Model": mname, **m})

robust_df = pd.DataFrame(robust_rows)
robust_df.to_csv(f"{OUTPUT_DIR}/temporal_robustness.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["MAE", "RMSE"]):
    for mname, grp in robust_df.groupby("Model"):
        ax.plot(grp["Month"], grp[metric], marker="o", label=mname, linewidth=1.5)
    ax.set_title(f"Monthly Walk-Forward {metric}", fontsize=12)
    ax.set_xlabel("Month"); ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig_temporal_robustness.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅  Saved temporal_robustness.csv | fig_temporal_robustness.png")

# ── 5. Peak vs Off-Peak ───────────────────────────────────────────────────────
q75 = np.percentile(y_test, 75)
peak_mask = y_test >= q75
peak_rows = []
for mname, p in mean_preds.items():
    for cond, mask in [("Peak", peak_mask), ("Off-Peak", ~peak_mask)]:
        peak_rows.append({"Model": mname, "Condition": cond,
                          "Avg_Traffic": round(y_test[mask].mean(), 2),
                          "MAE": round(mean_absolute_error(y_test[mask], p[mask]), 4)})
pd.DataFrame(peak_rows).to_csv(f"{OUTPUT_DIR}/peak_offpeak_results.csv", index=False)
print(f"✅  Saved peak_offpeak_results.csv")

# ── 6. Terminal-level analysis ────────────────────────────────────────────────
df_raw = pd.read_parquet(DATA_PATH)
col_dt = "movement.scheduledTime.local" if "movement.scheduledTime.local" in df_raw.columns \
         else "movement.scheduledTime.utc"
df_raw["scheduled_dt"] = pd.to_datetime(df_raw[col_dt], errors="coerce")
if df_raw["scheduled_dt"].dt.tz is not None:
    df_raw["scheduled_dt"] = df_raw["scheduled_dt"].dt.tz_convert(None)
df_raw["date_hour"] = df_raw["scheduled_dt"].dt.floor("h")

test_start  = pd.to_datetime(dates_test[0])
test_end    = pd.to_datetime(dates_test[-1])
test_raw    = df_raw[(df_raw["date_hour"] >= test_start) &
                     (df_raw["date_hour"] <= test_end) &
                     df_raw["movement.terminal"].notna()].copy()

best_p    = mean_preds[best_model]
date_pred = pd.Series(best_p, index=pd.to_datetime(dates_test))
total_ts  = pd.Series(y_test,  index=pd.to_datetime(dates_test))

term_rows = []
for term, grp in test_raw.groupby("movement.terminal"):
    term_ts = grp.groupby("date_hour").size()
    common  = term_ts.index.intersection(pd.to_datetime(dates_test))
    if len(common) < 10:
        continue
    ratio = (term_ts.loc[common] / total_ts.loc[common].clip(lower=1)).values
    p_t   = (date_pred.loc[common] * ratio).values
    m     = metrics(term_ts.loc[common].values, p_t)
    term_rows.append({"Terminal": str(term), **m, "n_hours": len(common)})

term_df = pd.DataFrame(term_rows).sort_values("Terminal")
term_df.to_csv(f"{OUTPUT_DIR}/terminal_results.csv", index=False)
print(f"✅  Saved terminal_results.csv")
print(term_df.to_string(index=False))

# ── 7. Residual distribution figure (RF+DW vs RF+N2V) ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mname, color in zip(axes[:2],
                             ["RF+DW",  "RF+N2V"],
                             ["#4CAF50", "#2196F3"]):
    residuals = y_test - mean_preds[mname]
    ax.hist(residuals, bins=50, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(0, color="red", linestyle="--", linewidth=1.2)
    ax.set_title(f"Residuals – {mname}", fontsize=11)
    ax.set_xlabel("Residual"); ax.set_ylabel("Count")
ax = axes[2]
ax.boxplot([y_test - mean_preds["RF+DW"], y_test - mean_preds["RF+N2V"]],
           labels=["RF+DW", "RF+N2V"], patch_artist=True,
           boxprops=dict(facecolor="#E8F5E9"))
ax.axhline(0, color="red", linestyle="--", linewidth=1.2)
ax.set_title("Residual Boxplot: RF+DW vs RF+N2V", fontsize=11)
ax.set_ylabel("Residual")
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig_residuals.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅  Saved fig_residuals.png")

# ── 8. Training and Inference Timing ─────────────────────────────────────────
print("\nMeasuring training and inference time (seed=42) ...")
set_seed(42)

TIME_MODELS = {
    "RF":             ("rf",  TEMP_COLS),
    "XGBoost":        ("xgb", TEMP_COLS),
    "LightGBM":       ("lgb", TEMP_COLS),
    "RF+N2V":         ("rf",  FEAT_COLS_MAP["RF+N2V"]),
    "XGBoost+N2V":    ("xgb", FEAT_COLS_MAP["XGBoost+N2V"]),
    "LightGBM+N2V":   ("lgb", FEAT_COLS_MAP["LightGBM+N2V"]),
    "RF+DW":          ("rf",  FEAT_COLS_MAP["RF+DW"]),
    "XGBoost+DW":     ("xgb", FEAT_COLS_MAP["XGBoost+DW"]),
    "LightGBM+DW":    ("lgb", FEAT_COLS_MAP["LightGBM+DW"]),
}

timing_rows = []
for model_name, (mtype, feat_cols) in TIME_MODELS.items():
    X_tr_t = trainval_df[feat_cols].values
    y_tr_t = trainval_df[TARGET].values
    X_te_t = test_df[feat_cols].values
    n_v    = int(len(X_tr_t) * 0.15)

    t0 = _time.time()
    if mtype == "xgb":
        mdl = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=1, reg_alpha=1,
                           early_stopping_rounds=20, eval_metric="rmse",
                           random_state=42, verbosity=0)
        mdl.fit(X_tr_t[:-n_v], y_tr_t[:-n_v],
                eval_set=[(X_tr_t[-n_v:], y_tr_t[-n_v:])], verbose=False)
    elif mtype == "lgb":
        mdl = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                subsample=0.8, colsample_bytree=0.8,
                                reg_lambda=1, reg_alpha=1, random_state=42, verbose=-1)
        mdl.fit(X_tr_t[:-n_v], y_tr_t[:-n_v],
                eval_set=[(X_tr_t[-n_v:], y_tr_t[-n_v:])],
                callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)])
    else:
        mdl = RandomForestRegressor(n_estimators=300, max_depth=10,
                                    min_samples_leaf=2, n_jobs=-1, random_state=42)
        mdl.fit(X_tr_t, y_tr_t)
    t_train = _time.time() - t0

    t0 = _time.time()
    _ = mdl.predict(X_te_t)
    t_infer_ms = (_time.time() - t0) / len(X_te_t) * 1000

    timing_rows.append({
        "Model":                   model_name,
        "n_features":              len(feat_cols),
        "train_time_s":            round(t_train, 2),
        "inference_ms_per_sample": round(t_infer_ms, 4),
    })
    print(f"  {model_name:15s} | features={len(feat_cols):3d} | "
          f"train={t_train:.2f}s | infer={t_infer_ms:.4f}ms")

pd.DataFrame(timing_rows).to_csv(f"{OUTPUT_DIR}/timing_models.csv", index=False)
print(f"\n✅  Saved timing_models.csv")

print("\n" + "="*60)
print("ALL EVALUATION COMPLETE")
print("="*60)
print(f"Outputs saved to: {OUTPUT_DIR}")

Best model: RF+DW  (MAE=1.7862)
  DM  XGBoost+N2V vs XGBoost       stat=-6.969  p=0.0000  ✓
  DM  XGBoost+N2V vs RF            stat=-1.528  p=0.1266  ✗
  DM  XGBoost+N2V vs LightGBM      stat=-8.798  p=0.0000  ✓
  DM  XGBoost+N2V vs SVR           stat=-6.438  p=0.0000  ✓
  DM  XGBoost+N2V vs Ridge         stat=-10.370  p=0.0000  ✓
  DM  XGBoost+N2V vs MLP           stat=-14.225  p=0.0000  ✓
  DM  XGBoost+N2V vs LSTM          stat=-6.023  p=0.0000  ✓
  DM  XGBoost+N2V vs Persistence   stat=-11.418  p=0.0000  ✓
  DM  XGBoost+N2V vs ARIMA         stat=-20.984  p=0.0000  ✓
  DM  RF+N2V vs XGBoost       stat=-4.544  p=0.0000  ✓
  DM  RF+N2V vs RF            stat=-2.896  p=0.0038  ✓
  DM  RF+N2V vs LightGBM      stat=-5.325  p=0.0000  ✓
  DM  RF+N2V vs SVR           stat=-4.559  p=0.0000  ✓
  DM  RF+N2V vs Ridge         stat=-8.180  p=0.0000  ✓
  DM  RF+N2V vs MLP           stat=-9.949  p=0.0000  ✓
  DM  RF+N2V vs LSTM          stat=-6.062  p=0.0000  ✓
  DM  RF+N2V vs Persistence   stat=-9.6